In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.market_math import prepare_pair_parameters, compute_log_spread, stable_seed
from src.convergence_signal import calculate_convergence_signal


# 04 Conditional fOU Convergence Signal
Preview the same conditional first-passage forecast used inside the daily backtest.


In [ ]:
cfg = ResearchConfig()


In [ ]:
train_prices = pd.read_parquet("train_prices.parquet")
test_prices = pd.read_parquet("test_prices.parquet")
full_prices = pd.concat([train_prices, test_prices])
params = prepare_pair_parameters(
    pd.read_parquet("eligible_pairs.parquet"),
    pd.read_parquet("cointegrated_pairs.parquet"),
).sort_values("pair")
EVALUATION_DATE = test_prices.index[0]
print("Signal snapshot date:", EVALUATION_DATE.date())


In [ ]:
rows = []
curves = {}
for row in params.itertuples():
    history = compute_log_spread(
        full_prices, row.dependent, row.independent, row.alpha, row.beta
    ).loc[:EVALUATION_DATE]
    z = (history.iloc[-1] - row.mu) / np.sqrt(row.variance)
    if abs(z) < cfg.entry_z:
        continue

    signal, curve = calculate_convergence_signal(
        history,
        row.mu,
        row.kappa,
        row.sigma,
        row.hurst,
        row.variance,
        cfg.target_probability,
        cfg.entry_z,
        cfg.memory_window,
        cfg.max_horizon_days,
        cfg.n_paths,
        seed=stable_seed(cfg.seed, row.pair, EVALUATION_DATE),
    )
    if signal["selected_dte_trading_days"] is None:
        continue

    rows.append({
        "pair": row.pair,
        "dependent": row.dependent,
        "independent": row.independent,
        "beta": row.beta,
        "signal_date": EVALUATION_DATE,
        "direction": signal["direction"],
        "z": z,
        "horizon": signal["selected_dte_trading_days"],
        "probability": signal["probability_at_selected_dte"],
    })
    curves[row.pair] = curve

snapshot = pd.DataFrame(rows)
snapshot.to_parquet("signal_snapshot.parquet")
pd.DataFrame(curves).to_parquet("signal_probability_curves.parquet")
display(snapshot)

if curves:
    pd.DataFrame(curves).iloc[:, :5].plot(figsize=(10, 4), title="Conditional first-passage probabilities")
    plt.axhline(cfg.target_probability, color="black", linestyle="--")
    plt.show()
